# spaCR mask generator (Google Colab)

Masks from a spaCR model, run by the real spaCR pipeline on Google's free GPU under your own Google account.
The result is the same ZIP as the [spaCR mask generator page](https://einarolafsson.github.io/projects/spacr/#mask-generator):
the image, the label mask, the flows, the cell probability (when the model returns them), `objects.csv`, a README and a manifest.

1. For a GPU, choose **Runtime > Change runtime type > T4 GPU** (free). Without one the notebook still runs on the CPU, slowly.
2. Choose **Runtime > Run all**, or run the cells one by one.
3. Upload an image when asked, or leave `USE_TEST_IMAGE` ticked for a held-out test image from Hugging Face.

Your image stays in this Colab runtime; it is not sent to spaCR or its authors. Model weights are downloaded from Hugging Face and checked against their published SHA-256.

## 1. Install spaCR

Downloads spaCR from GitHub and installs it with its dependencies (a few minutes). If Colab asks to restart the session, restart it and run the cells from step 2 on.

In [ ]:
SPACR_BRANCH = "nightly"  #@param {type:"string"}
import os, sys
if not os.path.isdir("/content/spacr"):
    !git clone --quiet --depth 1 --branch {SPACR_BRANCH} https://github.com/EinarOlafsson/spacr.git /content/spacr
%pip install --quiet "/content/spacr[plaque]" "pypdfium2>=5.13.0,<6"
sys.path.insert(0, "/content/spacr")

## 2. Choose the model and the settings

- **MODEL_CLASS**: what to segment. `toxoplasma_well_detector` returns one box per well instead of a mask.
- **MODEL_VERSION**: `default` is the version the spaCR desktop app uses.
- **DIAMETER**: object diameter in pixels; blank uses the magnification rule (`MAGNIFICATION`); `auto` lets the model choose (native scale for plaques).
- **FLOW_THRESHOLD** and **CELLPROB_THRESHOLD**: Cellpose thresholds; a blank cell probability threshold keeps the model's default.
- **CHANNELS**: zero-based channel numbers, for example `0` or `0,1,2`; blank uses channel 0 (all three of an RGB image for plaques and wells).

In [ ]:
MODEL_CLASS = "toxoplasma_pv"  #@param ["toxoplasma_pv", "toxoplasma_plaque", "toxoplasma_well_detector", "cell_from_hoechst", "nuclei_from_cellmask", "toxoplasma_from_cellmask", "toxoplasma_from_hoechst", "live_cell"]
MODEL_VERSION = "default"  #@param {type:"string"}
DIAMETER = ""  #@param {type:"string"}
MAGNIFICATION = 20  #@param {type:"number"}
FLOW_THRESHOLD = 0.4  #@param {type:"number"}
CELLPROB_THRESHOLD = ""  #@param {type:"string"}
CHANNELS = ""  #@param {type:"string"}
DETECTION_CONFIDENCE = 0.25  #@param {type:"number"}
USE_TEST_IMAGE = True  #@param {type:"boolean"}

import os, sys
sys.path.insert(0, "/content/spacr")
from tools.web_mask_generator import notebook as spacr_web
print(spacr_web.device_report())
default_key, versions = spacr_web.model_versions(MODEL_CLASS)
print("Model class:", MODEL_CLASS, "| versions:", ", ".join(versions), "| default:", default_key)

## 3. Choose an image

TIFF and OME-TIFF (multi-channel), PNG, JPEG, BMP and PDF; at most 64 MB.

In [ ]:
from pathlib import Path
os.makedirs("/content/input", exist_ok=True)
if USE_TEST_IMAGE:
    IMAGE, credit = spacr_web.random_example(MODEL_CLASS, folder="/content/input")
    print(credit)
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    IMAGE = Path("/content/input") / name
    IMAGE.write_bytes(uploaded[name])
print("Image:", IMAGE)

## 4. Run

Downloads the model once (about 1.2 GB for segmentation models, 5 MB for the well detector), runs spaCR and draws the result.

In [ ]:
model = spacr_web.fetch_model(MODEL_CLASS, MODEL_VERSION, cache_dir="/content/spacr-models")
os.makedirs("/content/output", exist_ok=True)
RESULT = spacr_web.generate(IMAGE, MODEL_CLASS, model, output_dir="/content/output", diameter=DIAMETER,
                            flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD,
                            channels=CHANNELS, magnification=MAGNIFICATION,
                            detection_confidence=DETECTION_CONFIDENCE)
info = spacr_web.summary(RESULT)
print("Found:", info["count"], "| model:", model["key"])
print("ZIP contents:", ", ".join(info["files"]))
spacr_web.show(RESULT)

## 5. Download the ZIP

To try other settings, change them in step 2 and run steps 2 and 4 again (the model is not downloaded again).

In [ ]:
spacr_web.offer_download(RESULT)